Libraries

In [ ]:
import re
import math
from pathlib import Path
from typing import Optional, Dict, Iterable, List, Tuple, Any
import copy
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import nbinom, lognorm, skewnorm, exponweib
from scipy.optimize import minimize
from scipy.stats import (
    poisson,
    nbinom,
    lognorm,
    gamma as gamma_dist,
    weibull_min,
    skewnorm,
    genpareto,
    burr12,
    kstest,
)

Get Files

In [ ]:
FILE_PATH = "/Users/ben/Desktop/Uni/ACTL/ACTL4001/Assignment Information/srcsc-2026-claims-cargo.xlsx"   # <-- change if needed
FREQ_SHEET = "freq"
SEV_SHEET = "sev"

Cleaning Rules

In [ ]:
# -------------------------------------------------------------------
# Validation rules based on the Cargo Loss Claims data dictionary
# -------------------------------------------------------------------
NUMERIC_RULES = {
    "weight": (1_500, 250_000),
    "route_risk": (1, 5),
    "distance": (1, 100),
    "transit_duration": (1, 60),
    "pilot_experience": (1, 30),
    "vessel_age": (1, 50),
    "solar_radiation": (0, 1),
    "debris_density": (0, 1),
    "exposure": (0, 1),
    "claim_count": (0, 5),
    "claim_seq": (0, 5),
}

MONEY_RULES_RAW = {
    "cargo_value": (50_000, 680_000_000),
    "claim_amount": (31_000, 678_000_000),
}

NA_LIKE = {"", "na", "n/a", "null", "none", "missing", "nan", "-", "--", "?"}
BAD_PLACEHOLDER_PATTERNS = ["_???", "???"]

COLUMN_ALIASES = {
    "cointainer_type": "container_type",
    "container": "container_type",
    "claim_amount1": "claim_amount",
    "claim amount": "claim_amount",
    "cargo value": "cargo_value",
    "route risk": "route_risk",
    "transit duration": "transit_duration",
    "pilot experience": "pilot_experience",
    "vessel age": "vessel_age",
    "solar radiation": "solar_radiation",
    "debris density": "debris_density",
}

LOWERCASE_CATEGORICAL_COLS = {"origin", "destination", "cargo_type", "container_type"}
INTEGER_LIKE_COLS = {"route_risk", "claim_count"}
STRICT_INTEGER_COLS = {"claim_seq", "claim_count"}  

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)


Cleaning Function

In [ ]:
def clean_column_name(col: str) -> str:
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return COLUMN_ALIASES.get(col, col)


def normalise_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [clean_column_name(c) for c in out.columns]
    return out


def add_log(log_rows: List[dict], sheet: str, action: str, rows_removed: int = 0, details: str = "") -> None:
    log_rows.append(
        {
            "sheet": sheet,
            "action": action,
            "rows_removed": int(rows_removed),
            "details": details,
        }
    )


def standardise_strings(df: pd.DataFrame, sheet: str, log_rows: List[dict]) -> pd.DataFrame:
    out = df.copy()
    object_like_cols = list(out.select_dtypes(include=["object", "string"]).columns)
    cells_changed = 0

    for col in object_like_cols:
        before = out[col].astype("string")
        s = before.str.strip().str.replace(r"\s+", " ", regex=True)

        na_mask = s.str.lower().isin(NA_LIKE)
        s = s.mask(na_mask, pd.NA)

        if col in LOWERCASE_CATEGORICAL_COLS:
            s = s.str.lower()

        out[col] = s
        after = out[col].astype("string")
        cells_changed += int((before.fillna("<NA>") != after.fillna("<NA>")).sum())

    add_log(log_rows, sheet, "Standardised text fields", 0, f"{cells_changed} cell(s) changed")
    return out


def coerce_numeric_columns(df: pd.DataFrame, numeric_cols: Iterable[str], sheet: str, log_rows: List[dict]) -> pd.DataFrame:
    out = df.copy()

    for col in numeric_cols:
        if col not in out.columns:
            continue

        before_na = out[col].isna().sum()
        out[col] = pd.to_numeric(out[col], errors="coerce")
        after_na = out[col].isna().sum()

        new_nas = int(after_na - before_na)
        if new_nas > 0:
            add_log(log_rows, sheet, f"Coerced {col} to numeric", 0, f"{new_nas} value(s) became NaN")

    return out


def infer_money_scale(series: pd.Series, lower_raw: float, upper_raw: float) -> Tuple[float, Dict[float, float]]:
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return 1.0, {1.0: np.nan, 1_000.0: np.nan, 1_000_000.0: np.nan}

    candidate_scales = [1.0, 1_000.0, 1_000_000.0]
    fit_scores = {}

    for scale in candidate_scales:
        scaled = values * scale
        fit_scores[scale] = float(((scaled >= lower_raw) & (scaled <= upper_raw)).mean())

    best_scale = max(fit_scores, key=fit_scores.get)
    return best_scale, fit_scores


def standardise_money_columns(df: pd.DataFrame, money_rules_raw: Dict[str, Tuple[float, float]], sheet: str, log_rows: List[dict]) -> pd.DataFrame:
    out = df.copy()

    for col, (low_raw, high_raw) in money_rules_raw.items():
        if col not in out.columns:
            continue

        scale, scores = infer_money_scale(out[col], low_raw, high_raw)

        if scale != 1.0:
            out[col] = pd.to_numeric(out[col], errors="coerce") * scale
            add_log(
                log_rows,
                sheet,
                f"Standardised monetary scale for {col}",
                0,
                f"multiplied by {scale:,.0f}; fit scores={scores}",
            )
        else:
            add_log(
                log_rows,
                sheet,
                f"Checked monetary scale for {col}",
                0,
                f"kept as-is; fit scores={scores}",
            )

    return out


def drop_rows_with_bad_placeholders(df: pd.DataFrame, sheet: str, log_rows: List[dict]) -> pd.DataFrame:
    out = df.copy()
    object_like_cols = list(out.select_dtypes(include=["object", "string"]).columns)

    if not object_like_cols:
        add_log(log_rows, sheet, "Dropped rows containing bad placeholders", 0, "No object/string columns found")
        return out

    bad_mask = pd.Series(False, index=out.index)

    for col in object_like_cols:
        s = out[col].astype("string")
        col_bad = pd.Series(False, index=out.index)

        for pat in BAD_PLACEHOLDER_PATTERNS:
            col_bad = col_bad | s.str.contains(re.escape(pat), na=False)

        bad_mask = bad_mask | col_bad

    removed = int(bad_mask.sum())
    out = out.loc[~bad_mask].copy()

    add_log(
        log_rows,
        sheet,
        "Dropped rows containing bad placeholders",
        removed,
        f"patterns checked: {BAD_PLACEHOLDER_PATTERNS}",
    )
    return out


def drop_rows_with_any_missing(df: pd.DataFrame, sheet: str, log_rows: List[dict]) -> pd.DataFrame:
    out = df.copy()
    missing_mask = out.isna().any(axis=1)
    removed = int(missing_mask.sum())
    out = out.loc[~missing_mask].copy()

    add_log(log_rows, sheet, "Dropped rows with any missing value", removed)
    return out


def drop_exact_duplicates(df: pd.DataFrame, sheet: str, log_rows: List[dict]) -> pd.DataFrame:
    out = df.copy()
    dup_mask = out.duplicated(keep="first")
    removed = int(dup_mask.sum())
    out = out.loc[~dup_mask].copy()

    add_log(log_rows, sheet, "Dropped exact duplicate rows", removed)
    return out


def drop_out_of_range(df: pd.DataFrame, rules: Dict[str, Tuple[float, float]], sheet: str, log_rows: List[dict]) -> pd.DataFrame:
    out = df.copy()

    for col, (low, high) in rules.items():
        if col not in out.columns:
            continue

        mask = out[col].notna() & ((out[col] < low) | (out[col] > high))
        removed = int(mask.sum())
        out = out.loc[~mask].copy()

        add_log(log_rows, sheet, f"Dropped out-of-range rows for {col}", removed, f"valid range: [{low}, {high}]")

    return out


def cast_integer_like_columns(df: pd.DataFrame, cols: Iterable[str], sheet: str, log_rows: List[dict]) -> pd.DataFrame:
    out = df.copy()

    for col in cols:
        if col not in out.columns:
            continue

        non_null = out[col].dropna()
        if non_null.empty:
            continue

        frac_mask = (non_null % 1 != 0)
        bad_frac = int(frac_mask.sum())

        if bad_frac > 0:
            add_log(log_rows, sheet, f"Rounded non-integer values in {col}", 0, f"{bad_frac} row(s) rounded")

        out[col] = out[col].round().astype("Int64")

    return out

def drop_non_integer_rows(df: pd.DataFrame, cols: Iterable[str], sheet: str, log_rows: List[dict]) -> pd.DataFrame:
    out = df.copy()

    for col in cols:
        if col not in out.columns:
            continue

        mask = out[col].notna() & (out[col] % 1 != 0)
        removed = int(mask.sum())
        out = out.loc[~mask].copy()

        add_log(
            log_rows,
            sheet,
            f"Dropped non-integer rows for {col}",
            removed,
            "only integer values allowed",
        )

        if col in out.columns:
            out[col] = out[col].astype("Int64")

    return out


def clean_sheet(df: pd.DataFrame, sheet_name: str) -> Tuple[pd.DataFrame, List[dict], dict]:
    log_rows: List[dict] = []
    before_rows = len(df)

    out = normalise_columns(df)
    add_log(log_rows, sheet_name, "Normalised column names", 0, f"columns={list(out.columns)}")

    out = standardise_strings(out, sheet_name, log_rows)

    numeric_cols = set(NUMERIC_RULES) | set(MONEY_RULES_RAW)
    out = coerce_numeric_columns(out, numeric_cols, sheet_name, log_rows)

    out = standardise_money_columns(out, MONEY_RULES_RAW, sheet_name, log_rows)

    out = drop_rows_with_bad_placeholders(out, sheet_name, log_rows)
    out = drop_rows_with_any_missing(out, sheet_name, log_rows)
    out = drop_exact_duplicates(out, sheet_name, log_rows)

    out = drop_non_integer_rows(out, STRICT_INTEGER_COLS, sheet_name, log_rows)

    out = drop_out_of_range(out, NUMERIC_RULES, sheet_name, log_rows)
    out = drop_out_of_range(out, MONEY_RULES_RAW, sheet_name, log_rows)

    out = cast_integer_like_columns(out, INTEGER_LIKE_COLS, sheet_name, log_rows)

    after_rows = len(out)
    before_after = {
        "sheet": sheet_name,
        "rows_before": before_rows,
        "rows_after": after_rows,
        "rows_removed_total": before_rows - after_rows,
    }

    return out.reset_index(drop=True), log_rows, before_after


def build_summary(log_rows: List[dict], before_after: List[dict]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    actions_df = pd.DataFrame(log_rows)
    before_after_df = pd.DataFrame(before_after)

    if actions_df.empty:
        actions_df = pd.DataFrame(columns=["sheet", "action", "rows_removed", "details"])

    if before_after_df.empty:
        before_after_df = pd.DataFrame(columns=["sheet", "rows_before", "rows_after", "rows_removed_total"])

    return actions_df, before_after_df


def print_section(title: str) -> None:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def main() -> None:
    file_path = Path(FILE_PATH)
    if not file_path.exists():
        raise FileNotFoundError(f"Input file not found: {file_path}")

    print_section("READING INPUT FILE")
    print(f"Input file: {file_path}")

    freq_raw = pd.read_excel(file_path, sheet_name=FREQ_SHEET)
    sev_raw = pd.read_excel(file_path, sheet_name=SEV_SHEET)

    print(f"{FREQ_SHEET}: {len(freq_raw):,} rows loaded")
    print(f"{SEV_SHEET}: {len(sev_raw):,} rows loaded")

    freq_clean, freq_log, freq_before_after = clean_sheet(freq_raw, FREQ_SHEET)
    sev_clean, sev_log, sev_before_after = clean_sheet(sev_raw, SEV_SHEET)

    combined_log = freq_log + sev_log
    before_after_rows = [freq_before_after, sev_before_after]

    actions_df, before_after_df = build_summary(combined_log, before_after_rows)
    actions_df = actions_df[["sheet", "action", "rows_removed", "details"]]
    before_after_df = before_after_df[["sheet", "rows_before", "rows_after", "rows_removed_total"]]

    output_path = Path("/Users/ben/Desktop/Uni/ACTL/ACTL4001/Assignment Code/Basic Data Cleaning and Modeling/srcsc-2026-claims-cargo_cleaned.xlsx")
    
    print_section("CLEANING ACTIONS")
    if len(actions_df) == 0:
        print("No cleaning actions recorded.")
    else:
        print(actions_df.to_string(index=False))

    print_section("BEFORE / AFTER ROW COUNTS")
    print(before_after_df.to_string(index=False))

    print_section("WRITING CLEAN FILE")
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        freq_clean.to_excel(writer, sheet_name=FREQ_SHEET, index=False)
        sev_clean.to_excel(writer, sheet_name=SEV_SHEET, index=False)
        before_after_df.to_excel(writer, sheet_name="row_summary", index=False)
        actions_df.to_excel(writer, sheet_name="cleaning_summary", index=False)

    print(f"Cleaned workbook saved to: {output_path}")

    print_section("FINAL OUTPUT")
    print(f"{FREQ_SHEET}: {len(freq_raw):,} -> {len(freq_clean):,} rows")
    print(f"{SEV_SHEET}: {len(sev_raw):,} -> {len(sev_clean):,} rows")
    print(f"Total rows removed: {(len(freq_raw) - len(freq_clean)) + (len(sev_raw) - len(sev_clean)):,}")


if __name__ == "__main__":
    main()